# EDA 01 — Orders table (late delivery focus)

Source: `olist.silver.orders` — the silver layer is now populated (built Nov 18 2026), including the derived target `is_late` and the lead-time columns `promised_days` / `actual_days`.

Coverage: Sep 2016 – Oct 2018 (public Olist dataset snapshot), 99,441 orders.

Key questions:
* How many orders are delivered late vs the estimated delivery date?
* How is the delivery delay distributed, and does it vary by purchase month?
* Does the silver `is_late` flag agree with a direct date comparison? (transformation sanity check)

In [0]:
# C1 — Table overview: rows, uniqueness, date coverage, nulls
profile_df = spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT order_id) AS distinct_orders,
  COUNT(DISTINCT customer_id) AS distinct_customers,
  MIN(order_purchase_timestamp) AS earliest_purchase,
  MAX(order_purchase_timestamp) AS latest_purchase,
  COUNT_IF(order_approved_at IS NULL) AS null_approved,
  COUNT_IF(order_delivered_carrier_date IS NULL) AS null_carrier_date,
  COUNT_IF(order_delivered_customer_date IS NULL) AS null_delivered_customer,
  COUNT_IF(order_estimated_delivery_date IS NULL) AS null_estimated,
  COUNT_IF(is_late IS NULL) AS null_is_late,
  MAX(_processed_at) AS silver_processed_at
FROM olist.silver.orders
""")
display(profile_df)

In [0]:
# C2 — Order status distribution
status_df = spark.sql("""
SELECT order_status, COUNT(*) AS orders,
  ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM olist.silver.orders), 2) AS pct
FROM olist.silver.orders
GROUP BY order_status
ORDER BY orders DESC
""")
display(status_df)

In [0]:
# C3 — Delivery delay stats (delivered orders): actual vs estimated
# Uses the silver-derived is_late flag and lead-time columns;
# islate_mismatches cross-checks is_late against raw date math (must be 0)
delay_stats_df = spark.sql("""
WITH d AS (
  SELECT
    is_late,
    actual_days,
    DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) AS delay_days
  FROM olist.silver.orders
  WHERE order_status = 'delivered'
)
SELECT
  COUNT(*) AS delivered_orders,
  COUNT_IF(is_late = 1) AS late_orders,
  ROUND(100.0 * COUNT_IF(is_late = 1) / COUNT(*), 2) AS pct_late,
  ROUND(AVG(delay_days), 2) AS avg_delay_days,
  ROUND(percentile(delay_days, 0.5), 2) AS median_delay_days,
  MIN(delay_days) AS min_delay,
  MAX(delay_days) AS max_delay,
  ROUND(AVG(actual_days), 2) AS avg_leadtime_days,
  COUNT_IF(delay_days > 15) AS very_late_over_15d,
  COUNT_IF(is_late IS DISTINCT FROM CAST(delay_days > 0 AS INT)) AS islate_mismatches
FROM d
""")
display(delay_stats_df)

In [0]:
# C4 — Delay distribution buckets
delay_bucket_df = spark.sql("""
WITH d AS (
  SELECT DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) AS delay_days
  FROM olist.silver.orders
  WHERE order_status = 'delivered'
)
SELECT
  CASE
    WHEN delay_days <= -20 THEN 'very early (<= -20d)'
    WHEN delay_days BETWEEN -19 AND -10 THEN 'early (-19 to -10d)'
    WHEN delay_days BETWEEN -9 AND -1 THEN 'early (-9 to -1d)'
    WHEN delay_days = 0 THEN 'on time (0d)'
    WHEN delay_days BETWEEN 1 AND 5 THEN 'late (1-5d)'
    WHEN delay_days BETWEEN 6 AND 15 THEN 'late (6-15d)'
    WHEN delay_days > 15 THEN 'very late (>15d)'
  END AS delay_bucket,
  MIN(delay_days) AS sort_key,
  COUNT(*) AS orders,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM d
GROUP BY 1
ORDER BY sort_key
""")
display(delay_bucket_df)

In [0]:
# C5 — Monthly late-delivery trend
monthly_df = spark.sql("""
WITH d AS (
  SELECT
    DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) AS delay_days,
    date_trunc('month', order_purchase_timestamp) AS purchase_month
  FROM olist.silver.orders
  WHERE order_status = 'delivered'
)
SELECT purchase_month,
  COUNT(*) AS delivered,
  COUNT_IF(delay_days > 0) AS late,
  ROUND(100.0 * COUNT_IF(delay_days > 0) / COUNT(*), 1) AS pct_late,
  ROUND(AVG(delay_days), 1) AS avg_delay_days
FROM d
GROUP BY purchase_month
ORDER BY purchase_month
""")
display(monthly_df)

## Findings (re-run Nov 18 2026 on `olist.silver.orders`)

* **99,441 orders**, all unique (`order_id` and `customer_id` 1:1) — silver processed 2026-11-18 14:35 UTC.
* **97.02% delivered** (96,478); `shipped` 1.1%, `canceled` 0.63%, `unavailable` 0.61%, remainder in-flight statuses.
* **6.77% of delivered orders arrive late** (6,534 / 96,478) per the silver `is_late` flag — identical to the bronze-based run. **The flag is validated:** 0 mismatches vs direct date math across all 96,478 delivered orders.
* Estimates are systematically conservative: median delay **-12 days** (early), average -11.88d, range -147d early to +188d late; average actual lead time 12.5 days. Buckets: 17.0% very early (≤-20d), 47.1% early (-19 to -10d), 27.8% early (-9 to -1d), 1.3% on time, 6.8% late — of which 1.27% very late (>15d).
* **Monthly spikes:** late rate jumps to 12.4% (Nov 2017), 14.1% (Feb 2018), **19.0% (Mar 2018)** vs ~3% 2017 baseline; best month Jun 2018 (1.2%). (Sep 2016 shows 100% but n=1.)
* **Data quirk surfaced by silver:** 8 delivered orders have **no delivery date** (null `is_late`), and 6 non-delivered orders have one — 2,971 rows total have null `is_late` and must be excluded/handled in modeling.
* Modeling implication: class imbalance (~7% late) needs attention; purchase month/seasonality is a strong predictor candidate.